In [ ]:
# imports


import numpy as np
import matplotlib.pyplot as plt
# import idk

import json



this whole thing is claim based therefore 

the datasets fever and scifact make sense for cross dataset testing (both claim based)

wiki changed and article claim testing

similar structure therefore model should be able to work on both

In [24]:
# data loading respectively

# training on fever since its significantly larger
# validation on fever
# testing on scifact

In [ ]:
LABEL_MAP_FEVER = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT ENOUGH INFO": 2,
}

def load_fever(path, wiki_sentences=None, max_samples=None):
    """
    path: path to FEVER jsonl
    wiki_sentences: optional dict {(page, sent_id): sentence_text}
    """

    data = []
    with open(path, "r") as f:
        for i, line in enumerate(f):
            ex = json.loads(line)

            claim = ex["claim"]
            label = LABEL_MAP_FEVER.get(ex["label"], None)
            if label is None:
                continue

            evidence_text = ""

            # Try to reconstruct evidence text if wiki sentences provided
            if label != 2 and wiki_sentences is not None:
                evidence_sets = ex.get("evidence", [])
                sentences = []

                for ev_set in evidence_sets:
                    for ev in ev_set:
                        if len(ev) >= 2:
                            page, sent_id = ev[0], ev[1]
                            key = (page, sent_id)
                            if key in wiki_sentences:
                                sentences.append(wiki_sentences[key])

                evidence_text = " ".join(sentences)

            data.append({
                "claim": claim,
                "evidence": evidence_text,
                "label": label
            })

            if max_samples and len(data) >= max_samples:
                break

    return data

In [ ]:
def load_scifact_corpus(path):
    corpus = {}

    with open(path, "r") as f:
        for line in f:
            doc = json.loads(line)
            doc_id = doc["doc_id"]
            sentences = doc["abstract"]  # list of sentences
            corpus[doc_id] = sentences

    return corpus

In [ ]:
LABEL_MAP_SCIFACT = {
    "SUPPORT": 0,
    "CONTRADICT": 1,
    "NOINFO": 2,
}

def load_scifact(claims_path, corpus, max_samples=None):
    data = []

    with open(claims_path, "r") as f:
        for i, line in enumerate(f):
            ex = json.loads(line)

            claim = ex["claim"]
            evidence = ex.get("evidence", {})

            # Default: NEI
            label = 2
            evidence_text = ""

            if evidence:
                # take first annotated doc
                for doc_id, annotations in evidence.items():
                    doc_id = int(doc_id)
                    doc_sentences = corpus.get(doc_id, [])

                    for ann in annotations:
                        label_str = ann["label"]
                        label = LABEL_MAP_SCIFACT[label_str]

                        sent_ids = ann.get("sentences", [])
                        selected = [
                            doc_sentences[i] for i in sent_ids
                            if i < len(doc_sentences)
                        ]

                        evidence_text = " ".join(selected)
                        break
                    break

            data.append({
                "claim": claim,
                "evidence": evidence_text,
                "label": label
            })

            if max_samples and len(data) >= max_samples:
                break

    return data

In [ ]:
# FEVER
fever_data = load_fever("fever_train.jsonl", wiki_sentences=None)

# SciFact
corpus = load_scifact_corpus("corpus.jsonl")
scifact_data = load_scifact("claims_train.jsonl", corpus)

print(fever_data[0])
print(scifact_data[0])